# Multiplicative fBM noise: signature control vs the fully recurrent network of Horvath, Teichmann & Žurič

This notebook isolates the last case of the paper: $N=D=K=2$, a 2-D fractional Brownian motion with $H=0.30$, and **multiplicative** noise ($\sigma_2\neq0$). No closed-form optimum is known for this case.

We compare two ways of computing a non-anticipative control from **the same 10,000 simulated fBM paths**:
* **the signature method** at $(L,M)=(7,3)$: the closed-form minimiser of the signature proxy $\mathtt J^{\rm L}$, with empirical functionals built from the training paths;
* **the fully recurrent network (fRNN)** of Horvath, Teichmann & Žurič, "Deep hedging under rough volatility" (*Risks*, 2021). It was designed for hedging under non-Markovian rough volatility; here it is adapted to optimal control. At each time $t_i$ it reads the value of the fBM, $W_{t_i}$, and a hidden state $\tilde\delta_{t_i}$, and returns the control and the next hidden state. It uses no signatures and no model structure.

Both controls are functions of the noise path only, with no state feedback. They are compared **with each other**, on the same test paths with common randomness, and by **running time**.

All library code is in the package `lqsig` in `src/` of this repository, shared with the main notebook `01_signature_lq_control.ipynb`. Figures are written to `figures/frnn_benchmark/` and cached results to `cache/`.

**Running in two stages (compute, then plot).** The first code cell has three settings:
* `MODE = "compute"` runs every computation and saves the results of each expensive section to `CACHE_FILE`, a pickle file in the repository's `cache/` folder, after each section finishes. These are the signature fit, the trained network weights and its training log, the test evaluation and the timings.
* `MODE = "load"` skips those computations, reads the results from `CACHE_FILE`, and only prints the tables and draws the figures. Cheap steps, such as the noise paths, the gradient check and the sample-path figure, are recomputed in both modes.
* `USE_TEX` switches LaTeX rendering of the figures on or off.

So on a fast machine without LaTeX, run with `MODE = "compute"`, `USE_TEX = False`, then copy the cache file into `cache/` of the repository on the LaTeX machine and run with `MODE = "load"`, `USE_TEX = True`. All reported timings are those of the compute machine. The load run checks that the settings match the cached run.

In [ ]:
# ================= run settings =================
# Two-stage workflow:
#   1. on the fast machine (no LaTeX):  MODE = "compute", USE_TEX = False  -> runs everything, saves results to the cache
#   2. on the LaTeX machine:            MODE = "load",    USE_TEX = True   -> reads the cache, only prints and plots
# MODE = "compute" with USE_TEX = True also works (one machine, everything at once).
MODE = "compute"                         # "compute" or "load"
USE_TEX = True                           # False on machines without a LaTeX installation
# =================================================

import os, sys, time, gc
import numpy as np
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = "retina"

def _find_repo(start):
    """The repository root: the first folder above `start` that contains src/lqsig."""
    p = os.path.abspath(start)
    while not os.path.isdir(os.path.join(p, "src", "lqsig")):
        if os.path.dirname(p) == p:
            raise FileNotFoundError("cannot find the repository root (the folder that contains src/lqsig)")
        p = os.path.dirname(p)
    return p

REPO = _find_repo(os.getcwd())
if os.path.join(REPO, "src") not in sys.path:          # not needed after `pip install -e .`
    sys.path.insert(0, os.path.join(REPO, "src"))
from lqsig import *                                    # the shared library (src/lqsig)

np.set_printoptions(precision=5, suppress=True, linewidth=120)
setup_matplotlib(USE_TEX)
FIG_DIR = set_figure_dir(os.path.join(REPO, "figures", "frnn_benchmark"))
CACHE_FILE = os.path.join(REPO, "cache", "frnn_benchmark.pkl")
_results = ResultCache(MODE, CACHE_FILE)
cache_save, cache_load = _results.save, _results.load

# One seed for every random draw in the notebook: each experiment uses default_rng(SEED + its own offset).
SEED = 0
np.random.seed(SEED)

In [ ]:
# ---------------- settings ----------------
H, nF = 0.30, 400                 # Hurst index, time steps on [0, T]
L_sig, M_sig = 7, 3               # signature truncation levels
n_train, n_val, n_test = 10000, 1000, 4000
frnn_width = 12                   # units in each of the two hidden layers of every cell (as in Horvath et al., Figure 3.2)
frnn_train = dict(n_epochs=75, batch=256, lr=1e-2, lr_decay=lambda k: 1 / (1 + k / 1000), clip=10.0)   # 75 epochs as in the paper

# ---------------- model (as in the main notebook) ----------------
mm = LQModel(T=1.0, X0=[10.0, -5.0], b0=[2.0, 0.0],
             b1=[[1.0, 0.0], [0.2, 0.8]],                       # N x K
             b2=[[-0.3, 0.7], [0.5, -0.5]],                     # N x N
             sigma0=[[0.9, 0.1], [0.0, 1.0]],                   # N x D
             sigma2=[[[0.60, 0.0], [0.0, 0.4]],                 # N x D x N
                     [[0.1, 0.30], [-0.4, 0.2]]],
             A=[0*np.eye(2)], B=[0.5 * np.eye(2), 0.5 * np.eye(2)],   # B(t) = 0.5 (1 + t) I
             C=[[0.0, 0.0]], Dlin=[[0.0, 0.0]],
             E=np.eye(2), G=[0.0, 0.0])

# ---------------- noise ----------------
chol = fbm_cholesky(H, nF, mm.T); dt = mm.T / nF
dB_train = fbm_increments(H, n_train, nF, mm.D, mm.T, np.random.default_rng(SEED + 20), chol=chol)
dB_val = fbm_increments(H, n_val, nF, mm.D, mm.T, np.random.default_rng(SEED + 22), chol=chol)
dB_test = fbm_increments(H, n_test, nF, mm.D, mm.T, np.random.default_rng(SEED + 21), chol=chol)
timing = {}                       # wall-clock times in seconds
settings = dict(H=H, nF=nF, L_sig=L_sig, M_sig=M_sig, n_train=n_train, n_val=n_val, n_test=n_test, frnn_width=frnn_width,
                frnn_train={k: v for k, v in frnn_train.items() if k != 'lr_decay'})
if MODE == "compute":
    cache_save("settings", settings=settings)
else:
    assert cache_load("settings")['settings'] == settings, "the settings above differ from those of the cached run"
print(f"N = {mm.N}, D = {mm.D}, K = {mm.K};  H = {H}, {nF} steps;  paths: {n_train} train, {n_val} validation, {n_test} test")

## Signature method at $(L,M)=(7,3)$
* **Offline:** the empirical functionals from the training paths, then one closed-form fit.
* **Online:** the control $u_t=\langle\theta,\widehat{\mathbb W}^{\le M}_t\rangle$, evaluated along the test paths with the same Wong–Zakai step as everywhere else.

In [ ]:
if MODE == "compute":
    t0 = time.perf_counter()
    F_sig = EmpiricalCostFunctionals(mm, L_sig, M_sig, dB_train, dt, record_every=4)
    timing['sig_functionals'] = time.perf_counter() - t0
    t0 = time.perf_counter()
    fit_sig = solve_signature_lq(mm, L_sig, M_sig, F=F_sig)
    timing['sig_fit'] = time.perf_counter() - t0
    del F_sig; gc.collect()
    c_sig_val = simulate_driver(mm, dB_val, dt, signature_control(fit_sig['theta'], mm, M_sig), sig_level=M_sig)[0]
    cache_save("signature", fit_sig=fit_sig, c_sig_val=c_sig_val,
               t_functionals=timing['sig_functionals'], t_fit=timing['sig_fit'])
else:
    _c = cache_load("signature")
    fit_sig, c_sig_val = _c['fit_sig'], _c['c_sig_val']
    timing['sig_functionals'], timing['sig_fit'] = _c['t_functionals'], _c['t_fit']
ctrl_sig = signature_control(fit_sig['theta'], mm, M_sig)
print(f"empirical functionals (L={L_sig}, M={M_sig}, {n_train} paths): {timing['sig_functionals']:.1f}s")
print(f"closed-form fit ({fit_sig['theta'].size} coefficients): {timing['sig_fit']:.2f}s;  in-sample proxy value J^L = {fit_sig['value']:.5f}")
print(f"validation cost of the signature control: {fmt_mc(c_sig_val, 5)}")

## Fully recurrent network (Horvath, Teichmann & Žurič, 2021)

In [ ]:
# gradient check on a short problem: backpropagation through time vs central finite differences
_n = 20; _chol = fbm_cholesky(H, _n, mm.T)
_dB = fbm_increments(H, 32, _n, mm.D, mm.T, np.random.default_rng(SEED + 40), chol=_chol)
_pol = FRNNPolicy(mm, _n, width=5, seed=SEED + 41); _pol.W3 *= 3
_c, _g = frnn_cost_and_grad(_pol, _dB); _ps = _pol.params(); _rng = np.random.default_rng(SEED + 42); _err = 0.0
for j in range(len(_ps)):
    for _ in range(4):
        idx = tuple(_rng.integers(s) for s in _ps[j].shape); old = _ps[j][idx]
        _ps[j][idx] = old + 1e-6; cp = frnn_cost_and_grad(_pol, _dB, need_grad=False)[0].mean()
        _ps[j][idx] = old - 1e-6; cm = frnn_cost_and_grad(_pol, _dB, need_grad=False)[0].mean(); _ps[j][idx] = old
        fd = (cp - cm) / 2e-6; _err = max(_err, abs(fd - _g[j][idx]) / (abs(fd) + 1e-6))
_cs = simulate_driver(mm, _dB, mm.T / _n, frnn_control(_pol, _dB))[0]
print(f"largest relative gradient error: {_err:.1e};  cost vs simulate_driver: {np.abs(_cs - _c).max() / np.abs(_c).max():.1e}")

In [ ]:
pol = FRNNPolicy(mm, nF, width=frnn_width, seed=SEED + 32)
n_par = sum(p.size for p in pol.params())
print(f"fRNN: {nF} cells, each R^{2 * mm.D} -> {frnn_width} -> {frnn_width} -> R^{mm.K + mm.D};  {n_par} parameters in total")
if MODE == "compute":
    tr = train_frnn_policy(pol, dB_train, seed=SEED + 33, val=dB_val, **frnn_train)
    timing['frnn_train'] = tr['train_time']
    c_in = frnn_cost_and_grad(pol, dB_train[:2000], need_grad=False)[0]
    cache_save("frnn", params=[p.copy() for p in pol.params()], tr=tr, c_in=c_in, t_train=timing['frnn_train'])
else:
    _c = cache_load("frnn")
    pol.set_params([p.copy() for p in _c['params']]); tr, c_in = _c['tr'], _c['c_in']; timing['frnn_train'] = _c['t_train']
    for ep, cv in zip(tr['val_epoch'], tr['val_cost']):                # the training log of the cached run
        if ep > 0 and (ep % 5 == 0 or ep == 1 or ep == tr['val_epoch'][-1]):
            j = list(tr['val_epoch']).index(ep)
            print(f"epoch {ep:3d} ({ep * (tr['iterations'] // len(tr['batch_cost'])):5d} iterations)  mean batch cost {tr['batch_cost'][ep - 1]:9.4f}  "
                  f"training time {tr['val_time'][j]:6.0f}s  validation cost {cv:9.4f}")
print(f"training time (excluding validation passes): {timing['frnn_train']:.0f}s;  {tr['iterations']} iterations, "
      f"{frnn_train['n_epochs']} passes over the {n_train} training paths")
print(f"overfitting check: cost on 2,000 of its own training paths {fmt_mc(c_in, 4)}, on the validation paths {tr['val_cost'][-1]:.4f}")

## Out-of-sample comparison

In [ ]:
sig_name, net_name = f"signature L={L_sig}, M={M_sig}", "fRNN (Horvath et al.)"
ctrls = {sig_name: (ctrl_sig, M_sig), net_name: (frnn_control(pol, dB_test), 0)}
if MODE == "compute":
    ev = evaluate_controls(mm, ctrls, dB_test, dt, reference=net_name)     # the fRNN is the benchmark
    cache_save("evaluation", ev=ev)
else:
    ev = cache_load("evaluation")['ev']
for k, v in ev.items():
    print(f"{k:>24}: test cost {fmt_mc(v['cost'], 5)}")
d = ev[sig_name]['diff']                                               # J(signature) − J(fRNN), path by path
rel, rel_ci = rel_excess_ci(d, ev[sig_name]['cref'])
print(f"\nsignature − fRNN (paired): {fmt_mc(d, 5)}")
print(f"relative excess cost of the signature control over the fRNN: {rel:+.3f} ± {rel_ci:.3f}% [95% CI]"
      f"   (positive: the network is better)")
print(f"L2(dP⊗dt) distance between the two controls: {fmt_l2(ev[sig_name]['d2'], 4)}")

## Running time
* **Offline time** is the time to obtain the control from the 10,000 training paths.
  * Signature method: the empirical functionals plus the closed-form fit.
  * Network: training, excluding the validation passes.

  Neither count includes simulating the training paths, which both methods share.
* **Online time** is the time to compute the control along the 4,000 test paths.

In [ ]:
def time_control_only(ctrl, lev):
    t0 = time.perf_counter(); X0 = np.tile(mm.X0.astype(float), (n_test, 1))
    stream = signature_stream(dB_test, dt, lev) if lev > 0 else None
    for i in range(nF):
        ctrl(i, i * dt, X0, next(stream) if stream is not None else None)
    return time.perf_counter() - t0

if MODE == "compute":                     # timings are measured on the machine that does the computations
    online = {}
    for k, (ctrl, lev) in ctrls.items():
        t_c = time_control_only(ctrl, lev)
        t0 = time.perf_counter(); simulate_driver(mm, dB_test, dt, ctrl, sig_level=lev); t_s = time.perf_counter() - t0
        online[k] = (t_c, t_s)
    cache_save("online_timing", online=online)
else:
    online = cache_load("online_timing")['online']
offline = {sig_name: timing['sig_functionals'] + timing['sig_fit'], net_name: timing['frnn_train']}
print(f"{'control':>24} | {'offline [s]':>11} | {'online: control only [s]':>24} | {'online: full simulation [s]':>27} | test cost")
for k in ctrls:
    print(f"{k:>24} | {offline[k]:11.1f} | {online[k][0]:24.2f} | {online[k][1]:27.1f} | {fmt_mc(ev[k]['cost'], 4)}")
print(f"\nsignature offline time split: functionals {timing['sig_functionals']:.1f}s + fit {timing['sig_fit']:.2f}s")
d_val = [cv - c_sig_val for cv in tr['val_paths']]                 # paired with the signature control, per epoch
ep_match = next((int(e) for e, dd in zip(tr['val_epoch'], d_val) if dd.mean() <= 0), None)
best = int(np.argmin([dd.mean() for dd in d_val]))
print(f"best validation epoch: {tr['val_epoch'][best]} (paired difference to the signature control {fmt_mc(d_val[best], 4)}, "
      f"after {tr['val_time'][best]:.0f}s of training)")
if ep_match is None:
    print("the fRNN's validation cost never reached the signature control's validation cost")
else:
    j = list(tr['val_epoch']).index(ep_match)
    print(f"the fRNN first matched the signature control on the validation paths at epoch {ep_match}, "
          f"after {tr['val_time'][j]:.0f}s of training (paired difference {fmt_mc(d_val[j], 4)})")

In [ ]:
# the fRNN's validation cost during training, paired with the signature control on the same validation paths
fig, ax = plt.subplots(figsize=(10, 5.5))
dv = np.array([mc_mean(dd) for dd in d_val]); tv = np.maximum(tr['val_time'], 1.0)
ax.plot(tv, dv[:, 0], 'k.-', lw=1.2, label=r"$\mathrm{fRNN\ minus\ signature\ (paired,\ validation\ paths,\ after\ each\ epoch)}$")
ax.fill_between(tv, dv[:, 1], dv[:, 2], color='k', alpha=0.15, lw=0)
ax.axhline(0.0, color='C1', lw=1.5, label=rf"$\mathrm{{signature}}\ (L,M)=({L_sig},{M_sig})$")
ax.axvline(offline[sig_name], color='C1', ls='--', lw=1.2, label=r"$\mathrm{offline\ time\ of\ the\ signature\ control}$")
shown = dv[1:] if len(dv) > 1 else dv                                   # the untrained start is far off scale
lo, hi = min(shown[:, 1].min(), 0) - 0.3, max(np.percentile(shown[:, 2], 90), 0) + 0.3
ax.set_ylim(lo, hi)
ax.set_xscale('log'); ax.set_xlabel(r"$\mathrm{training\ /\ offline\ time\ [s]}$")
ax.set_ylabel(r"$J(u^{\mathrm{fRNN}})-J(u^{7,3})$")
ax.set_title(rf"$\mathrm{{fBM}},\ H={H:g},\ \sigma_2\neq0{{:}}\ \mathrm{{fRNN\ training\ vs\ signature}}$")
ax.legend(fontsize=11, loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=1, frameon=False)
fig.tight_layout(); save_figure(fig, f"frnn_vs_signature_H{H:.2f}".replace(".", "")); plt.show()

## Reading the results

In [ ]:
path = 0                                  # which test path to plot (0, 1 or 2 with n_show = 3)
n_show = 3
dB_show = dB_test[:n_show]

# fRNN first, signature second: at equal zorder the later curve is drawn on top
ctrls_show = {r"$\mathrm{fRNN}$": (frnn_control(pol, dB_show), 0),
              rf"$\mathrm{{signature}}\ (L,M)=({L_sig},{M_sig})$": (ctrl_sig, M_sig)}

sims_show = compare_paths(mm, dB_show, dt, ctrls_show, simulate_driver, path=path,
                          title=rf"$\mathrm{{fBM}},\ H={H:g},\ N=D=K=2,\ \mathrm{{multiplicative\ noise}}$",
                          fname=f"frnn_vs_signature_sample_paths_H{H:.2f}".replace(".", "") + f"_path{path}")